# Avance 4 · Regresión de la potencia en la fase 2

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este notebook mide la potencia de una molécula (pActividad) en lugar de predecir si está
activa o no. El objetivo es ordenar los candidatos, para quedarse con los que más potencia
tienen sobre una proteína.


## 1. Preparación

In [1]:
import os
# Un solo hilo por modelo, como en el notebook 04: así la columna de tiempos es comparable.
os.environ["OMP_NUM_THREADS"] = "1"

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import mean_absolute_error, matthews_corrcoef, roc_auc_score, root_mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import make_scorer

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn):
    print(f"{m.__name__:<8} {m.__version__}")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESADOS = RAIZ / "data" / "processed"

sys.path.insert(0, str(RAIZ))   # el notebook corre desde notebooks/, la raíz no está en la ruta
from src.evaluation.regresion import metricas_regresion

sistema  Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.41
python   3.12.15
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1


### 2. Los datos

La unidad es el par compuesto-diana. Se usan los mismos pares de entrenamiento y validación
que la clasificación del notebook 04, y la partición de prueba no se toca. La diana entra como
columnas de 0 y 1, porque sin ella el modelo no sabe contra qué se midió.

De esos pares, solo los que tienen potencia exacta sirven para entrenar una regresión sencilla.
Los que solo traen cotas se cargan igual, en `limite_inferior` y `limite_superior`, porque la
concordancia de Harrell sí puede usarlos al evaluar aunque no se entrene con ellos.


In [2]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc2 = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = (pares[pares["activo_10um"].notna() & pares["particion"].ne("prueba")]   # la prueba no se toca
     .merge(desc2, on="smiles").reset_index(drop=True))
cols = [c for c in desc2.columns if c != "smiles"]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
y_pot = d["pActividad"].to_numpy()                  # la potencia; NaN si el par solo tiene cotas
y_cls = d["activo_10um"].astype(int).to_numpy()     # la etiqueta, para comparar con la clasificación
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()               # solo_chembl, solo_npass, mixto
lim_inf = d["limite_inferior"].to_numpy()           # cotas del intervalo, para la concordancia
lim_sup = d["limite_superior"].to_numpy()
exacta = ~np.isnan(y_pot)
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta · {(~exacta).sum():,} solo con cotas")

54,369 pares · 46,092 con potencia exacta · 8,277 solo con cotas


## 3. Seis modelos de regresión, más un piso

| Clasificación (notebook 04) | Regresión (aquí) |
|:--|:--|
| logística | `Ridge` (regresión lineal) |
| Naive Bayes | `BayesianRidge` (regresión bayesiana) |
| árbol | `DecisionTreeRegressor` |
| kNN | `KNeighborsRegressor` |
| SVM-RBF | `SVR` |
| MLP | `MLPRegressor` |

`media` aparece en la tabla de resultados pero **no es uno de los seis**, es el piso: predice
siempre el promedio, así que es la marca de «no aprender nada». Un modelo que no le gane no
aprendió nada útil.

**Esto es un piso optimista, no un resultado.** Los seis se entrenan solo con las mediciones
exactas, y eso deja fuera la mitad de los negativos: de los 15,554 pares inactivos, 7,536
—el 48.5%— existen únicamente como cotas, un «> 10 µM» sin número. Son justo los compuestos
flojos, así que el modelo aprende de un mundo más fácil que el real. Quien use la censura es
el baseline del 03b, y la comparación entre los dos es la que aísla cuánto cuesta tirarlos.

Cada modelo se evalúa con la misma validación cruzada agrupada del notebook 04 (5 pliegues por
`grupo_cv`, semillas 0, 1 y 2, que también fijan el azar de cada modelo).


In [3]:
MODELOS = {
    "media":  DummyRegressor(),                                          # el piso: no aprende nada
    "ridge":  make_pipeline(MinMaxScaler(), Ridge()),
    "bayes_ridge": make_pipeline(MinMaxScaler(), BayesianRidge()),
    "arbol":  DecisionTreeRegressor(min_samples_leaf=5),
    "knn":    make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "svr":    make_pipeline(MinMaxScaler(), SVR()),
    "mlp":    make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
SEMILLAS = (0, 1, 2)
CORTE = 5.0   # pActividad 5 = 10 µM, el mismo corte de la clasificación


def medir(lim_inf, lim_sup, y_cls, pred):
    """Las tres métricas de la decisión 2.9, más MCC y AUROC de la clase derivada."""
    return {**metricas_regresion(lim_inf, lim_sup, pred),
            "mcc": matthews_corrcoef(y_cls, pred >= CORTE),
            "auroc": roc_auc_score(y_cls, pred)}


filas = []
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)
    for k, (ajuste, val) in enumerate(cv.split(X, y_cls, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]           # se entrena solo con las exactas
        for nombre, modelo in MODELOS.items():
            m = clone(modelo)
            # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
            m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
            reloj = time.perf_counter()
            m.fit(X[ajuste_ex], y_pot[ajuste_ex])
            seg = time.perf_counter() - reloj
            pred = m.predict(X[val])
            fila = {"modelo": nombre, "semilla": semilla, "pliegue": k, "seg": seg}
            filas.append({**fila, "estrato": "todos",
                **medir(lim_inf[val], lim_sup[val], y_cls[val], pred)})
            for e in np.unique(procedencia[val]):
                sel = procedencia[val] == e
                ys = y_cls[val][sel]
                if min(ys.sum(), (1 - ys).sum()) >= 20:      # estrato con 20 casos de cada clase
                    filas.append({**fila, "estrato": e,
                        **medir(lim_inf[val][sel], lim_sup[val][sel], ys, pred[sel])})
res = pd.DataFrame(filas)

/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


...

/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],
/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/repo/src/evaluation/regresion.py:89: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e],


/usr/local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(


La métrica principal es la **concordancia de Harrell**, fijada antes de ver los resultados.
Es la única de las tres que usa también las mediciones censuradas, así que describe todos los
pares y no solo la mitad exacta. Es además la que permite comparar contra el piso del `03b`
(concordancia 0.700). El Spearman y el RMSE se reportan al lado, estos dos son solo sobre exactas.


In [4]:
def tabla(res, estrato="todos"):
    """Media por modelo, ordenada por concordancia. `desv_mcc` es dispersión entre pliegues, no error estándar."""
    r = res[res["estrato"] == estrato]
    t = r.groupby("modelo")[["concordancia", "spearman_exactas", "rmse_exactas",
        "mcc", "auroc", "seg"]].mean()
    t.insert(3, "desv_mcc", r.groupby("modelo")["mcc"].std())
    return t.sort_values("concordancia", ascending=False).round(3)

tabla_todos = tabla(res)
tabla_todos

,concordancia,spearman_exactas,rmse_exactas,desv_mcc,mcc,auroc,seg
modelo,,,,,,,
svr,0.742,0.641,1.034,0.016,0.350,0.818,140.135
knn,0.741,0.658,1.014,0.015,0.343,0.803,0.045
mlp,0.738,0.642,1.135,0.022,0.372,0.801,51.912
bayes_ridge,0.698,0.529,1.148,0.025,0.221,0.774,1.017
ridge,0.698,0.529,1.147,0.025,0.220,0.774,0.101
arbol,0.673,0.484,1.308,0.021,0.291,0.717,2.862
media,0.500,NaN,1.338,0.000,0.000,0.500,0.018


### 4. Por procedencia

Un promedio puede esconder cómo le va a los naturales, que son los que este proyecto existe
para priorizar. En clasificación ya se había visto: en el Avance 3 el bosque bajaba de 0.858
de AUROC en los pares de solo ChEMBL a 0.611 en los de solo NPASS. Aquí se revisa si la
regresión repite esa caída.


In [5]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["concordancia", "spearman_exactas", "rmse_exactas"]]

concordancia  spearman_exactas  rmse_exactas
procedencia modelo                                                   
solo_chembl knn                 0.736             0.642         1.018
            svr                 0.734             0.622         1.040
            mlp                 0.733             0.629         1.134
            bayes_ridge         0.687             0.503         1.150
            ridge               0.687             0.502         1.150
            arbol               0.668             0.469         1.311
            media               0.500               NaN         1.325
mixto       mlp                 0.584             0.249         1.142
            svr                 0.566             0.228         0.983
            knn                 0.563             0.282         0.985
            arbol               0.542             0.128         1.259
            ridge               0.518             0.062         1.091
            bayes_ridge         0.516             0.059         1.091
            media               0.500               NaN         1.499
solo_npass  mlp                 0.551             0.260         1.150
            svr                 0.537             0.313         0.816
            bayes_ridge         0.530             0.256         1.103
            ridge               0.528             0.256         1.050
            arbol               0.502             0.181         1.240
            media               0.500               NaN         1.638
            knn                 0.469             0.362         0.866

## 5. Ajuste fino de los dos mejores

In [6]:
# Regla fijada antes de correr: los dos de mayor concordancia, sin contar el piso.
DOS = list(tabla_todos.drop("media").sort_values("concordancia", ascending=False).index[:2])
print("los dos elegidos:", DOS)

los dos elegidos: ['svr', 'knn']


In [7]:
REJILLAS = {
    "ridge":       {"ridge__alpha": [0.1, 1, 10]},
    "bayes_ridge": {"bayesianridge__alpha_init": [None, 1.0]},
    "arbol":       {"max_depth": [6, None], "min_samples_leaf": [1, 5]},
    "knn":         {"kneighborsregressor__n_neighbors": [15, 30],
                    "kneighborsregressor__weights": ["uniform", "distance"]},
    "svr":         {"svr__C": [1, 10]},
    "mlp":         {"mlpregressor__alpha": [1e-4, 1e-2]},
}
SEMILLAS_AF = (0,)   # una sola semilla: con tres, afinar el SVR se pasa de dos horas

# El criterio interno es el orden, que es lo que mide la concordancia; dentro del pliegue
# solo hay exactas, así que se usa Spearman como su equivalente calculable.
ORDEN = make_scorer(lambda y, p: spearmanr(y, p).statistic)

filas_af = []
for semilla in SEMILLAS_AF:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)
    for k, (ajuste, val) in enumerate(cv.split(X, y_cls, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]
        for nombre in DOS:
            base = clone(MODELOS[nombre])
            base.set_params(**{p: semilla for p in base.get_params() if p.endswith("random_state")})
            m = GridSearchCV(base, REJILLAS[nombre], scoring=ORDEN, n_jobs=1,
                             cv=GroupKFold(3))     # agrupado, no estratificado: la y es continua
            reloj = time.perf_counter()
            m.fit(X[ajuste_ex], y_pot[ajuste_ex], groups=g[ajuste_ex])
            seg = time.perf_counter() - reloj
            pred = m.predict(X[val])
            fila = {"modelo": f"{nombre} afinado", "semilla": semilla, "pliegue": k,
                    "seg": seg, "elegidos": str(m.best_params_)}
            filas_af.append({**fila, "estrato": "todos",
                             **medir(lim_inf[val], lim_sup[val], y_cls[val], pred)})
            for e in np.unique(procedencia[val]):
                sel = procedencia[val] == e
                ys = y_cls[val][sel]
                if min(ys.sum(), (1 - ys).sum()) >= 20:
                    filas_af.append({**fila, "estrato": e,
                                     **medir(lim_inf[val][sel], lim_sup[val][sel], ys, pred[sel])})
res_af = pd.DataFrame(filas_af)
tabla(res_af)


,concordancia,spearman_exactas,rmse_exactas,desv_mcc,mcc,auroc,seg
modelo,,,,,,,
svr afinado,0.759,0.685,0.982,0.015,0.393,0.830,1235.567
knn afinado,0.749,0.677,0.989,0.017,0.355,0.809,38.480


In [8]:
res_af[res_af["estrato"] == "todos"].groupby("modelo")["elegidos"].value_counts()

modelo       elegidos                                                                            
knn afinado  {'kneighborsregressor__n_neighbors': 15, 'kneighborsregressor__weights': 'distance'}    5
svr afinado  {'svr__C': 10}                                                                          5
Name: count, dtype: int64

## 6. Registro de la corrida


In [9]:
# Registro en experimentos/ (decisión 2.9: clasificar con un corte o predecir la potencia).
sys.path.insert(0, str(RAIZ))          # así lo hace el notebook 03
from src.tracking import Corrida

with Corrida("baseline-regresion-sencilla-exactas", decision="2.9",
             params={"modelos": list(MODELOS), "pliegues": 5, "entrena_con": "solo exactas",
                     "objetivo": "pActividad", "corte_mcc": CORTE, "estratos": "procedencia",
                     "metrica_principal": "concordancia", "afinados": DOS,
                     "rejillas": {n: REJILLAS[n] for n in DOS},
                     "semillas_afinado": list(SEMILLAS_AF)},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()},
                "ajuste_fino": tabla(res_af).to_dict(orient="index")})

print(c.carpeta.name)

2026-10-26-antonio-2.9-baseline-regresion-sencilla-exactas


## 7. Lectura

# 8. Declaración de uso de IA y contribución
Qué hice — revisar y validar tus ediciones de código, redactar los textos de las celdas 0, 3, 5 y 8 sobre lo que tú decidiste, proponer los cuatro títulos.
Qué no hice — elegir los modelos, interpretar resultados, escribir la lectura o la declaración.
Qué validaste tú — ajustaste cada texto, elegiste BayesianRidge entre las tres opciones de Pame, y decidiste dejar media como piso.